# nuPlan Mini Scenario EDA

This notebook answers three questions before a larger IDM run:

1. How many distinct scenario anchors and scenario tags are in the 64 mini databases?
2. How are the anchors distributed across scenario types, maps, logs, and overlapping time windows?
3. What small, deterministic sample can be used for the next IDM pilot?

A catalog row represents one distinct anchor token. It is not one simulation timestep and it is not a materialized nuPlan scenario. The notebook writes two generated files under `artifacts/eda/`:

- `scenario_catalog.csv`: all eligible tagged anchors discovered with the devkit's valid-scene rule.
- `idm_mini_sample.csv`: up to five valid, mutually non-overlapping scenarios per official challenge type.


In [1]:
from __future__ import annotations

import hashlib
import os
import sqlite3
from pathlib import Path
from typing import Optional

import pandas as pd

SEED = 7
WINDOW_DURATION_S = 15.0
EXTRACTION_OFFSET_S = -3.0
SAMPLE_PER_TYPE = 5
IDM_BENCHMARK_SCENARIO_TYPES = [
    "starting_left_turn",
    "starting_right_turn",
    "starting_straight_traffic_light_intersection_traversal",
    "stopping_with_lead",
    "high_lateral_acceleration",
    "high_magnitude_speed",
    "low_magnitude_speed",
    "traversing_pickup_dropoff",
    "waiting_for_pedestrian_to_cross",
    "behind_long_vehicle",
    "stationary_in_traffic",
    "near_multiple_vehicles",
    "changing_lane",
    "following_lane_with_lead",
]


## 1. Locate the mini databases

The notebook uses `NUPLAN_HOST_DATA_ROOT` or `NUPLAN_DATA_ROOT` when set. For a normal local checkout, it also reads the repository's `.env` file without requiring another package.


In [2]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "README.md").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing .git and README.md")


def read_env_value(env_file: Path, name: str) -> Optional[str]:
    if not env_file.exists():
        return None
    for raw_line in env_file.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == name:
            return value.strip().strip(chr(34)).strip(chr(39))
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
env_file = PROJECT_ROOT / ".env"
data_root_value = (
    os.getenv("NUPLAN_HOST_DATA_ROOT")
    or os.getenv("NUPLAN_DATA_ROOT")
    or read_env_value(env_file, "NUPLAN_HOST_DATA_ROOT")
    or read_env_value(env_file, "NUPLAN_DATA_ROOT")
)
if data_root_value is None:
    raise RuntimeError("Set NUPLAN_HOST_DATA_ROOT or NUPLAN_DATA_ROOT before running this notebook")

DATA_ROOT = Path(data_root_value).expanduser().resolve()
MINI_ROOT = DATA_ROOT / "nuplan-v1.1" / "splits" / "mini"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_FILES = sorted(MINI_ROOT.glob("*.db"))
if not DB_FILES:
    raise FileNotFoundError(f"No mini databases found under {MINI_ROOT}")

print(f"Data root: {DATA_ROOT}")
print(f"Mini databases: {len(DB_FILES)}")
print(f"Output directory: {OUTPUT_DIR}")


Data root: /Users/jimhe/Documents/sjsu/DATA298/nuplan-dataset
Mini databases: 64
Output directory: /Users/jimhe/Documents/sjsu/DATA298/data298a-causal-inference-uplift/artifacts/eda


## 2. Original categories stored in the databases

Category names come directly from `scenario_tag.type`, linked to an anchor by `scenario_tag.lidar_pc_token`. This EDA reads existing dataset labels; it does not infer them from speed, steering, or IDM scores. The 14 names in `IDM_BENCHMARK_SCENARIO_TYPES` are our benchmark filter, not all categories in mini.

The following table queries all 64 databases **before** scene, goal, window, or route filtering. `tag_rows` counts database records; `tagged_anchors` counts distinct anchors carrying each tag within each log; `logs` counts databases containing that tag. An anchor may appear under several tags, so summing category counts does not give the number of unique scenarios.


In [3]:
ORIGINAL_TAG_QUERY = """
SELECT type AS scenario_tag,
       COUNT(*) AS tag_rows,
       COUNT(DISTINCT lidar_pc_token) AS tagged_anchors
FROM scenario_tag
GROUP BY type
"""

original_tag_frames = []
for db_file in DB_FILES:
    with sqlite3.connect(db_file) as connection:
        tags = pd.read_sql_query(ORIGINAL_TAG_QUERY, connection)
    tags["log_name"] = db_file.stem
    original_tag_frames.append(tags)

original_categories = (
    pd.concat(original_tag_frames, ignore_index=True)
    .groupby("scenario_tag")
    .agg(tag_rows=("tag_rows", "sum"),
         tagged_anchors=("tagged_anchors", "sum"),
         logs=("log_name", "nunique"))
    .sort_index()
)
original_categories["in_benchmark_filter"] = original_categories.index.isin(IDM_BENCHMARK_SCENARIO_TYPES)
print(f"Original category names in the databases: {len(original_categories)}")
print(f"Benchmark categories present: {original_categories['in_benchmark_filter'].sum()} of {len(IDM_BENCHMARK_SCENARIO_TYPES)}")
print(original_categories.to_string())


Original category names in the databases: 67
Benchmark categories present: 14 of 14
                                                       tag_rows tagged_anchors  logs  in_benchmark_filter
scenario_tag                                                                                             
accelerating_at_crosswalk                                    55             55    15                False
accelerating_at_stop_sign                                    53             53     9                False
accelerating_at_stop_sign_no_crosswalk                       24             24     3                False
accelerating_at_traffic_light                               386            386    34                False
accelerating_at_traffic_light_with_lead                      35             35     5                False
accelerating_at_traffic_light_without_lead                  351            351    31                False
behind_bike                                                   2     

## 3. Build one catalog row per anchor token

This query follows the nuPlan devkit's valid-scene rule: the anchor's scene must have at least two scenes before it and two after it. Multiple `scenario_tag` rows are combined into `all_scenario_tags`; `scenario_type` uses the same lexical `MAX(type)` rule as the devkit discovery query. The SQLite files remain unchanged.


In [4]:
SCENARIO_QUERY = """
WITH ordered_scenes AS (
    SELECT token, ROW_NUMBER() OVER (ORDER BY name ASC) AS row_num
    FROM scene
),
num_scenes AS (
    SELECT COUNT(*) AS cnt FROM scene
),
valid_scenes AS (
    SELECT ordered_scenes.token
    FROM ordered_scenes
    CROSS JOIN num_scenes
    WHERE ordered_scenes.row_num >= 3
      AND ordered_scenes.row_num < num_scenes.cnt - 1
)
SELECT
    LOWER(HEX(lp.token)) AS scenario_token,
    lp.timestamp AS anchor_timestamp_us,
    l.map_version AS map_name,
    MAX(st.type) AS scenario_type,
    GROUP_CONCAT(DISTINCT st.type) AS all_scenario_tags,
    COUNT(DISTINCT st.type) AS tag_count,
    CASE
        WHEN s.goal_ego_pose_token IS NOT NULL AND goal.token IS NOT NULL THEN 1
        ELSE 0
    END AS valid_mission_goal,
    CASE
        WHEN s.roadblock_ids IS NOT NULL
         AND TRIM(s.roadblock_ids) NOT IN ('', '[]') THEN 1
        ELSE 0
    END AS has_route_roadblocks
FROM lidar_pc AS lp
INNER JOIN scenario_tag AS st ON lp.token = st.lidar_pc_token
INNER JOIN lidar AS ld ON ld.token = lp.lidar_token
INNER JOIN log AS l ON ld.log_token = l.token
INNER JOIN valid_scenes AS vs ON lp.scene_token = vs.token
INNER JOIN scene AS s ON lp.scene_token = s.token
LEFT JOIN ego_pose AS goal ON s.goal_ego_pose_token = goal.token
GROUP BY lp.token, lp.timestamp, l.map_version, s.token
ORDER BY lp.timestamp ASC
"""

LOG_RANGE_QUERY = "SELECT MIN(timestamp) AS log_start_us, MAX(timestamp) AS log_end_us FROM lidar_pc"


def read_scenario_anchors(db_file: Path) -> pd.DataFrame:
    with sqlite3.connect(db_file) as connection:
        frame = pd.read_sql_query(SCENARIO_QUERY, connection)
        log_range = pd.read_sql_query(LOG_RANGE_QUERY, connection).iloc[0]

    frame.insert(0, "log_name", db_file.stem)
    frame.insert(1, "db_file", db_file.name)
    frame["log_start_us"] = int(log_range["log_start_us"])
    frame["log_end_us"] = int(log_range["log_end_us"])
    return frame


In [5]:
catalog = pd.concat([read_scenario_anchors(db_file) for db_file in DB_FILES], ignore_index=True)

catalog["all_scenario_tags"] = catalog["all_scenario_tags"].apply(
    lambda value: "|".join(sorted(set(str(value).split(","))))
)
benchmark_type_set = set(IDM_BENCHMARK_SCENARIO_TYPES)
catalog["benchmark_scenario_type"] = catalog["all_scenario_tags"].apply(
    lambda tags: max(set(tags.split("|")) & benchmark_type_set)
    if set(tags.split("|")) & benchmark_type_set
    else None
)
catalog["window_start_us"] = catalog["anchor_timestamp_us"] + int(EXTRACTION_OFFSET_S * 1_000_000)
catalog["window_end_us"] = catalog["window_start_us"] + int(WINDOW_DURATION_S * 1_000_000)
catalog["complete_window"] = (
    (catalog["window_start_us"] >= catalog["log_start_us"])
    & (catalog["window_end_us"] <= catalog["log_end_us"])
)
catalog["scenario_id"] = catalog["log_name"] + ":" + catalog["scenario_token"]

catalog = catalog.sort_values(["log_name", "window_start_us", "scenario_token"]).reset_index(drop=True)
catalog["previous_anchor_gap_s"] = (
    catalog.groupby("log_name")["anchor_timestamp_us"].diff() / 1_000_000
)
catalog["overlaps_previous_window"] = catalog["previous_anchor_gap_s"].lt(WINDOW_DURATION_S).fillna(False)

catalog["time_block_number"] = (
    (catalog["anchor_timestamp_us"] - catalog["log_start_us"])
    // int(WINDOW_DURATION_S * 1_000_000)
).astype(int)
catalog["time_block_id"] = (
    catalog["log_name"] + ":" + catalog["time_block_number"].astype(str).str.zfill(5)
)

catalog_path = OUTPUT_DIR / "scenario_catalog.csv"
catalog.to_csv(catalog_path, index=False)
print(f"Saved {len(catalog):,} anchor rows to {catalog_path}")


Saved 334,753 anchor rows to /Users/jimhe/Documents/sjsu/DATA298/data298a-causal-inference-uplift/artifacts/eda/scenario_catalog.csv


## 4. Filtered categories and compact EDA summaries

The difference between `valid tag rows` and `distinct anchors` measures multilabel duplication. `Benchmark candidate anchors` applies the official 14 challenge categories plus the valid-mission-goal rule. Overlap is reported separately because two different anchors can still describe much of the same 15-second driving interval.

`all_scenario_tags` preserves the original labels. `scenario_type` is the alphabetically last tag among all labels on a catalog anchor. `benchmark_scenario_type` is the alphabetically last tag among only the 14 matching benchmark labels. These single-label assignments mirror the discovery query's `MAX(type)` convention; they do not mean the last label is more important. During sampling, `scenario_type` is set to `benchmark_scenario_type`.


In [6]:
benchmark_catalog = catalog.loc[
    catalog["benchmark_scenario_type"].notna() & catalog["valid_mission_goal"].eq(1)
].copy()

overview = pd.Series({
    "database files discovered": len(DB_FILES),
    "database files with eligible tagged anchors": catalog["db_file"].nunique(),
    "valid tag rows": int(catalog["tag_count"].sum()),
    "distinct anchors": len(catalog),
    "original category names across all databases": len(original_categories),
    "assigned catalog categories after scene filtering": catalog["scenario_type"].nunique(),
    "anchors with multiple tags": int(catalog["tag_count"].gt(1).sum()),
    "anchors with complete 15-second windows": int(catalog["complete_window"].sum()),
    "anchors with valid mission goals": int(catalog["valid_mission_goal"].sum()),
    "tag rows on valid-goal anchors": int(catalog.loc[catalog["valid_mission_goal"].eq(1), "tag_count"].sum()),
    "benchmark candidate anchors": len(benchmark_catalog),
    "benchmark scenario types": benchmark_catalog["benchmark_scenario_type"].nunique(),
    "anchors overlapping the previous anchor": int(catalog["overlaps_previous_window"].sum()),
    "15-second time blocks": catalog["time_block_id"].nunique(),
})
overview.to_frame("value")


Out[6]: 
                                                    value
database files discovered                              64
database files with eligible tagged anchors            63
valid tag rows                                     701862
distinct anchors                                   334753
original category names across all databases           67
assigned catalog categories after scene filtering      50
anchors with multiple tags                         202301
anchors with complete 15-second windows            334753
anchors with valid mission goals                   166550
tag rows on valid-goal anchors                     351957
benchmark candidate anchors                         57992
benchmark scenario types                               14
anchors overlapping the previous anchor            334690
15-second time blocks                                1513


In [7]:
category_summary = (
    benchmark_catalog.groupby("benchmark_scenario_type")
    .agg(
        anchors=("scenario_id", "size"),
        logs=("log_name", "nunique"),
        maps=("map_name", "nunique"),
        time_blocks=("time_block_id", "nunique"),
        multi_tag_anchors=("tag_count", lambda values: int(values.gt(1).sum())),
        complete_windows=("complete_window", "sum"),
        valid_goals=("valid_mission_goal", "sum"),
    )
    .sort_values("anchors", ascending=False)
)
print("Assigned benchmark categories after scene and mission-goal filtering:")
display(category_summary)
print("Examples of original tags and the assigned benchmark category:")
display(benchmark_catalog.loc[
    benchmark_catalog["tag_count"].gt(1),
    ["scenario_token", "all_scenario_tags", "benchmark_scenario_type"],
].head(5))


Assigned benchmark categories after scene and mission-goal filtering:
Examples of original tags and the assigned benchmark category:


,anchors,logs,maps,time_blocks,multi_tag_anchors,complete_windows,valid_goals
benchmark_scenario_type,,,,,,,
stationary_in_traffic,20190,33,2,143,20190,20190,20190
high_magnitude_speed,20024,45,4,254,6966,20024,20024
traversing_pickup_dropoff,13312,14,1,76,13312,13312,13312
low_magnitude_speed,2032,39,4,87,920,2032,2032
near_multiple_vehicles,835,20,3,37,477,835,835
behind_long_vehicle,663,3,1,10,663,663,663
starting_left_turn,268,15,4,22,268,268,268
starting_straight_traffic_light_intersection_traversal,173,45,3,159,173,173,173
high_lateral_acceleration,160,15,4,30,90,160,160


,scenario_token,all_scenario_tags,benchmark_scenario_type
0,c7470ea392845bee,stationary|stationary_in_traffic,stationary_in_traffic
1,5b4ae78ab0c853ca,stationary|stationary_in_traffic,stationary_in_traffic
2,ded65d602e265773,stationary|stationary_in_traffic,stationary_in_traffic
3,cf1862614ebc5af6,stationary|stationary_in_traffic,stationary_in_traffic
4,fab4847f9e2d5d77,stationary|stationary_in_traffic,stationary_in_traffic


In [8]:
map_summary = (
    catalog.groupby("map_name")
    .agg(anchors=("scenario_id", "size"), logs=("log_name", "nunique"), types=("scenario_type", "nunique"))
    .sort_values("anchors", ascending=False)
)
block_summary = catalog.groupby("time_block_id").size().describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

print("Anchors by map:")
display(map_summary)
print("Anchors per 15-second time block:")
display(block_summary.to_frame("anchors_per_block"))


Anchors by map:
Anchors per 15-second time block:


,anchors,logs,types
map_name,,,
us-nv-las-vegas-strip,269710,47,44
us-ma-boston,24790,5,33
us-pa-pittsburgh-hazelwood,22361,5,32
sg-one-north,17892,6,27


,anchors_per_block
count,1513.000000
mean,221.251157
std,77.944520
min,1.000000
50%,236.000000
75%,300.000000
90%,300.000000
95%,301.000000
99%,301.000000
max,301.000000


## 5. Create a deterministic IDM pilot sample

The pilot candidates must match one of the official 14 challenge categories and have a complete time window, valid mission goal, and route roadblocks. Categories with fewer candidates are processed first. Within each category, a stable hash of the seed and scenario ID determines selection order. The selection rejects any candidate whose 15-second window overlaps a previously selected window from the same log.

This is a small local pilot—not an exact reproduction of the full official benchmark.


In [9]:
def stable_rank(scenario_id: str) -> int:
    value = f"{SEED}:{scenario_id}".encode("utf-8")
    return int(hashlib.sha256(value).hexdigest()[:16], 16)


candidates = benchmark_catalog.loc[
    benchmark_catalog["complete_window"]
    & benchmark_catalog["has_route_roadblocks"].eq(1)
].copy()
candidates["scenario_type"] = candidates["benchmark_scenario_type"]
candidates["sample_rank"] = candidates["scenario_id"].map(stable_rank)

type_order = candidates.groupby("scenario_type").size().sort_values().index
selected_indices = []
selected_windows_by_log = {}

for scenario_type in type_order:
    type_candidates = candidates.loc[candidates["scenario_type"].eq(scenario_type)].sort_values("sample_rank")
    selected_for_type = 0
    for row in type_candidates.itertuples():
        existing_windows = selected_windows_by_log.setdefault(row.log_name, [])
        overlaps_selected = any(
            row.window_start_us < selected_end and selected_start < row.window_end_us
            for selected_start, selected_end in existing_windows
        )
        if overlaps_selected:
            continue
        selected_indices.append(row.Index)
        existing_windows.append((row.window_start_us, row.window_end_us))
        selected_for_type += 1
        if selected_for_type == SAMPLE_PER_TYPE:
            break

sample_columns = [
    "scenario_id",
    "log_name",
    "db_file",
    "scenario_token",
    "anchor_timestamp_us",
    "scenario_type",
    "all_scenario_tags",
    "map_name",
    "time_block_id",
    "window_start_us",
    "window_end_us",
]
idm_sample = (
    candidates.loc[selected_indices, sample_columns]
    .sort_values(["scenario_type", "log_name", "anchor_timestamp_us"])
    .reset_index(drop=True)
)

sample_path = OUTPUT_DIR / "idm_mini_sample.csv"
idm_sample.to_csv(sample_path, index=False)
print(f"Saved {len(idm_sample):,} sampled scenarios to {sample_path}")


Saved 68 sampled scenarios to /Users/jimhe/Documents/sjsu/DATA298/data298a-causal-inference-uplift/artifacts/eda/idm_mini_sample.csv


In [10]:
sample_summary = (
    idm_sample.groupby("scenario_type")
    .agg(scenarios=("scenario_id", "size"), logs=("log_name", "nunique"), maps=("map_name", "nunique"))
    .sort_index()
)

def sample_has_overlap(sample: pd.DataFrame) -> bool:
    for _, log_sample in sample.groupby("log_name"):
        ordered = log_sample.sort_values("window_start_us")
        starts = ordered["window_start_us"].tolist()
        ends = ordered["window_end_us"].tolist()
        if any(starts[index] < ends[index - 1] for index in range(1, len(ordered))):
            return True
    return False


assert idm_sample["scenario_id"].is_unique
assert not sample_has_overlap(idm_sample)
assert idm_sample.groupby("scenario_type").size().le(SAMPLE_PER_TYPE).all()

print(f"Sampled scenario types: {idm_sample['scenario_type'].nunique()}")
print(f"Sampled logs: {idm_sample['log_name'].nunique()}")
sample_summary


Sampled scenario types: 14
Sampled logs: 38
Out[10]: 
                                                    scenarios  logs  maps
scenario_type                                                            
behind_long_vehicle                                         5     3     1
changing_lane                                               5     5     3
following_lane_with_lead                                    4     4     1
high_lateral_acceleration                                   5     5     2
high_magnitude_speed                                        5     5     1
low_magnitude_speed                                         5     5     2
near_multiple_vehicles                                      5     5     3
starting_left_turn                                          5     5     3
starting_right_turn                                         5     5     3
starting_straight_traffic_light_intersection_tr...          5     5     3
stationary_in_traffic                                     

## Interpretation and next step

Use the tables above to check whether some categories are rare, concentrated in a small number of logs, or dominated by overlapping windows. The committed `configs/scenarios/idm_mini_sample.csv` is a reviewed snapshot of this notebook's generated sample. Only after that deterministic pilot completes should the sample size be increased or a full benchmark run be attempted.
